# ColdStart Killer — Evaluation: Retrieval Quality Assessment

This notebook runs the full 3-layer evaluation pipeline to measure how well the hybrid search system retrieves relevant products.

| Layer | What it tests | MongoDB required? |
|---|---|---|
| **Layer 1** — Diagnostics | `detect_language()`, `extract_hard_filters()` | ❌ No |
| **Layer 2** — IR Metrics | NDCG, Recall, MRR, Precision across 5 variants | ✅ Yes (or use fake results) |
| **Layer 3** — Demo Readiness | Claim Status table, latency, recommendations | ✅ Yes (or use fake results) |

**Quick start:** Run all cells top-to-bottom. If you don't have MongoDB/Ollama, set `USE_FAKE_RESULTS = True` in the Configuration cell.


### About N/A Metrics

Metrics showing `N/A` mean one of:

1. **No relevance judgments yet** - `retrieval_judgments_seed.json` is empty, so NDCG/Recall/MRR cannot be computed from human labels.
2. **Denominator is zero** - for example, Recall when no relevant items are known, or cold-share metrics when no relevant items appear in top-K.
3. **Missing cold-start status** - `is_cold_item` is unknown for some results.

### How to fix N/A metrics

1. Run `python scripts/build_eval_pool.py --queries evaluation/queries/retrieval_queries_seed.json --out .runtime/evaluation/pool_seed --top-k 20`.
2. Label candidates in `judgment_pool.csv` with relevance scores `0-3`.
3. Run `python scripts/import_eval_judgments.py --csv .runtime/evaluation/pool_seed/judgment_pool.csv --out evaluation/judgments/retrieval_judgments_seed.json`.
4. Re-run evaluation to replace `N/A` with real metric values.

> Warning: when judgments are empty, this notebook can still smoke-test the pipeline, but quality claims remain `needs_more_evidence`.


In [1]:
# ============================================
# STARTUP CHECK - Run this cell first
# ============================================
import sys
import platform

print(f"Python version: {sys.version}")
print(f"Platform: {platform.platform()}")

checks = {}
for pkg in ["json", "csv", "pathlib", "dataclasses"]:
    try:
        __import__(pkg)
        checks[pkg] = "\u2705 OK"
    except ImportError:
        checks[pkg] = "\u274c Missing"

# Optional packages
for pkg in ["pymongo", "numpy"]:
    try:
        __import__(pkg)
        checks[pkg] = "\u2705 OK"
    except ImportError:
        checks[pkg] = "\u26a0\ufe0f Missing (optional for fake mode)"

print("\nPackage check:")
for pkg, status in checks.items():
    print(f"  {pkg}: {status}")

from pathlib import Path
env_exists = (Path.cwd() / ".env").exists() or (Path.cwd().parent / ".env").exists()
print(f"\n.env file: {'\u2705 Found' if env_exists else '\u26a0\ufe0f Missing (needed for MongoDB)'}")

Python version: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0

Package check:
  json: ✅ OK
  csv: ✅ OK
  pathlib: ✅ OK
  dataclasses: ✅ OK
  pymongo: ✅ OK
  numpy: ✅ OK

.env file: ✅ Found


## ⚙️ Cấu hình — Thay đổi tham số tại đây

- **`USE_FAKE_RESULTS`**: `True` = chạy không cần MongoDB/Ollama (smoke test). `False` = chạy production pipeline.
- **`TOP_K`**: Số kết quả trả về mỗi variant.
- **`VARIANTS`**: Danh sách variants cần chạy.
- **`K_VALUES`**: Các giá trị K để tính metrics (NDCG@K, Recall@K, etc.).


In [2]:
# ============================================================
# ⚙️ CONFIGURATION — Thay đổi ở đây
# ============================================================

# Set True nếu không có MongoDB/Ollama/BGE-M3 (chạy với dữ liệu giả)
USE_FAKE_RESULTS: bool = False

# Số kết quả tối đa mỗi variant
TOP_K: int = 10

# Variants cần so sánh
VARIANTS: list = [
    "title_only",       # Baseline yếu: regex trên title
    "vector_only",      # Chỉ HyPE vector search
    "bm25_only",        # Chỉ BM25 proposition search
    "hybrid_union",     # Production: vector + BM25 + bonuses
    "hybrid_no_cold_boost",  # Ablation: bỏ cold_start_boost
]

# K values cho metrics
K_VALUES: list = [1, 3, 5, 10]

# Binary relevance threshold (relevance >= threshold = relevant)
RELEVANCE_THRESHOLD: int = 2

# Paths
QUERIES_PATH = "evaluation/queries/retrieval_queries_seed.json"
PROBES_PATH = "evaluation/queries/diagnostic_probes.json"
JUDGMENTS_PATH = "evaluation/judgments/retrieval_judgments_seed.json"
OUTPUT_DIR = ".runtime/evaluation/notebook_run"

print("Configuration:")
print(f"  USE_FAKE_RESULTS: {USE_FAKE_RESULTS}")
print(f"  TOP_K: {TOP_K}")
print(f"  VARIANTS: {VARIANTS}")
print(f"  K_VALUES: {K_VALUES}")
print(f"  RELEVANCE_THRESHOLD: {RELEVANCE_THRESHOLD}")

Configuration:
  USE_FAKE_RESULTS: False
  TOP_K: 10
  VARIANTS: ['title_only', 'vector_only', 'bm25_only', 'hybrid_union', 'hybrid_no_cold_boost']
  K_VALUES: [1, 3, 5, 10]
  RELEVANCE_THRESHOLD: 2


## Cell A — Safe Imports

Import evaluation package. Không gọi MongoDB hay Ollama. Nếu cell này fail, kiểm tra `src/evaluation/` có đầy đủ không.


In [3]:
# Cell A - Safe imports only, no side effects
import json
import os
import sys
import time
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    get_ipython().run_line_magic("cd", str(ROOT.parent))
    ROOT = Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Import evaluation modules (no side effects)
from src.evaluation.contracts import RunConfig, validate_run_config
from src.evaluation.dataset import (
    load_diagnostic_probes,
    load_eval_queries,
    load_relevance_judgments,
    judgments_by_query,
)
from src.evaluation.diagnostics import run_diagnostic_probes, summarize_diagnostics
from src.evaluation.metrics import compute_query_metrics, aggregate_metrics
from src.evaluation.runner import run_evaluation
from src.evaluation.reporting import (
    decide_claim_status,
    generate_metrics_summary_md,
    write_evaluation_outputs,
)
from src.evaluation.variants import EVALUATION_VARIANTS

print("\u2705 Evaluation imports loaded successfully.")
print(f"Project root: {ROOT}")
print(f"Available variants: {EVALUATION_VARIANTS}")

d:\HCPDB\ColdStart_Killer
✅ Evaluation imports loaded successfully.
Project root: d:\HCPDB\ColdStart_Killer
Available variants: ('title_only', 'vector_only', 'bm25_only', 'hybrid_union', 'hybrid_no_cold_boost')


## Cell B — Load Data

Load 20 diagnostic probes, 50 retrieval queries, và judgments. Validation chạy tự động — nếu data sai sẽ raise `ContractValidationError`.


In [4]:
# Cell B - Load and validate all evaluation data
probes = load_diagnostic_probes(PROBES_PATH)
queries = load_eval_queries(QUERIES_PATH)
judgments = load_relevance_judgments(JUDGMENTS_PATH)

print(f"Diagnostic probes loaded: {len(probes)}")
print(f"Retrieval queries loaded: {len(queries)}")
print(f"Relevance judgments loaded: {len(judgments)}")

# Show query distribution by slice
slice_counts = {}
for q in queries:
    for s in q.slices:
        slice_counts[s] = slice_counts.get(s, 0) + 1

print("\nQuery distribution by slice:")
print("Slice | Count")
print("--- | ---:")
for s, c in sorted(slice_counts.items(), key=lambda x: -x[1]):
    print(f"{s} | {c}")

if not judgments:
    print("\n\u26a0\ufe0f No judgments loaded. Metrics will show N/A for relevance-based scores.")
    print("   Run `scripts/build_eval_pool.py` first, then label, then re-run.")

Diagnostic probes loaded: 20
Retrieval queries loaded: 50
Relevance judgments loaded: 2119

Query distribution by slice:
Slice | Count
--- | ---:
english | 32
beauty | 26
electronics | 24
vietnamese | 13
persona | 10
constraint | 9
compatibility | 9
price_filter | 8
occasion | 7
problem | 5
vietnamese_no_diacritic | 5
gift | 4


---

# Layer 1 — Diagnostic Probes

Kiểm tra `detect_language()` và `extract_hard_filters()` hoạt động đúng chưa. **Không cần MongoDB.**

Mỗi probe có `expected_status`:
- `pass` = phải đúng
- `known_risk` = sai nhưng biết trước (documented gap)
- `expected_unsupported` = feature chưa implement
- `error` = phải raise exception


In [5]:
# Layer 1: Run diagnostic probes
print("Running diagnostic probes...")
diag_results = run_diagnostic_probes(probes)
diag_summary = summarize_diagnostics(diag_results)

print(f"\n{'='*60}")
print("LAYER 1 — DIAGNOSTIC SUMMARY")
print(f"{'='*60}")
print(f"Total probes:       {diag_summary['total_probes']}")
print(f"Testable probes:    {diag_summary['testable_probes']}")
print(f"Passed:             {diag_summary['passed']}")
print(f"Failed:             {diag_summary['failed']}")
print(f"Known risk:         {diag_summary['known_risk']}")
print(f"Unsupported:        {diag_summary['expected_unsupported']}")
print(f"Skipped (dep):      {diag_summary['skipped_dependency_missing']}")
pr = diag_summary['pass_rate']
print(f"Pass rate:          {pr:.1%}" if pr is not None else "Pass rate:          N/A")
print(f"{'='*60}")

Running diagnostic probes...


d:\HCPDB\ColdStart_Killer\src\query_processor.py:22: RuntimeWarning: Python 3.14+ detected. torch/sentence-transformers may be unstable. Consider using Python 3.10-3.12 for stability.
  from .embeddings import embed_one
d:\HCPDB\ColdStart_Killer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 1/1 [00:00<00:00,  2.37it/s]


LAYER 1 — DIAGNOSTIC SUMMARY
Total probes:       20
Testable probes:    14
Passed:             14
Failed:             0
Known risk:         4
Unsupported:        2
Skipped (dep):      0
Pass rate:          100.0%


In [6]:
# Layer 1: Detailed probe results table
print("probe_id | type | query | expected | actual | detail")
print("--- | --- | --- | --- | --- | ---")
for r in diag_results:
    pid = r.get('probe_id', '')
    ptype = r.get('probe_type', '')
    query = r.get('raw_query', '')
    if len(query) > 40:
        query = query[:37] + '...'
    expected = r.get('expected_status', '')
    actual = r.get('status', '')
    detail = r.get('detail', '')
    if len(str(detail)) > 60:
        detail = str(detail)[:57] + '...'
    
    # Color-code status
    icon = '\u2705' if actual == 'pass' else ('\u26a0\ufe0f' if actual == 'known_risk' else ('\u2796' if actual in ('expected_unsupported', 'skipped_dependency_missing') else '\u274c'))
    print(f"{pid} | {ptype} | {query} | {expected} | {icon} {actual} | {detail}")

probe_id | type | query | expected | actual | detail
--- | --- | --- | --- | --- | ---
p001 | language_detection | moisturizing cream for dry skin | pass | ✅ pass | 
p002 | language_detection | kem dưỡng ẩm cho da khô | pass | ✅ pass | 
p003 | language_detection | kem duong am cho da kho | known_risk | ⚠️ known_risk | No-diacritic Vietnamese. detect_language() uses Unicode r...
p004 | language_detection | sạc nhanh usb c cho iphone 15 | pass | ✅ pass | 
p005 | language_detection | op lung samsung galaxy a14 | known_risk | ⚠️ known_risk | No-diacritic Vietnamese. Will classify as English.
p006 | price_filter | wireless charger under 300k | pass | ✅ pass | 
p007 | price_filter | tai nghe không dây dưới 500k | pass | ✅ pass | 
p008 | price_filter | serum vitamin C between 200k and 500k | pass | ✅ pass | 
p009 | price_filter | ốp điện thoại từ 100k đến 300k | known_risk | ⚠️ known_risk | Vietnamese price range with từ...đến pattern. KNOWN RISK:...
p010 | price_filter | phone case over 200k

---

# Layer 2 — IR Metrics & Variant Comparison

Chạy 50 queries qua tất cả variants, tính NDCG@K, Recall@K, MRR@K, Precision@K, HitRate@K, và cold-start metrics.

| Variant | Vai trò |
|---|---|
| `title_only` | Baseline yếu — regex trên `items.title_en` |
| `vector_only` | Chỉ HyPE vector search |
| `bm25_only` | Chỉ BM25 proposition search |
| `hybrid_union` | **Main system** — production pipeline |
| `hybrid_no_cold_boost` | Ablation — bỏ `COLD_START_BOOST` |


In [7]:
# Layer 2: Build config and run full evaluation
import subprocess

try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=ROOT, stderr=subprocess.DEVNULL,
    ).decode().strip()
except Exception:
    git_commit = "unknown"

config = RunConfig(
    run_id=f"notebook_{'fake' if USE_FAKE_RESULTS else 'live'}",
    queries_path=QUERIES_PATH,
    judgments_path=JUDGMENTS_PATH,
    output_dir=OUTPUT_DIR,
    variants=VARIANTS,
    top_k=TOP_K,
    k_values=K_VALUES,
    relevance_threshold=RELEVANCE_THRESHOLD,
    use_cached_fixtures=False,
    created_at=time.strftime("%Y-%m-%dT%H:%M:%S"),
    git_commit=git_commit,
    plan_version="1.0",
    code_version="1.0",
    python_version=platform.python_version(),
    platform=platform.platform(),
)

validate_run_config(config)
print(f"Run ID: {config.run_id}")
print(f"Variants: {', '.join(config.variants)}")
print(f"Fake results: {USE_FAKE_RESULTS}")
print(f"\nStarting evaluation...")

t_start = time.perf_counter()
run_data = run_evaluation(
    config=config,
    queries=queries,
    judgments=judgments,
    use_fake_results=USE_FAKE_RESULTS,
)
t_end = time.perf_counter()

n_results = len(run_data.get('results', []))
n_failures = len(run_data.get('failures', []))
print(f"\n\u2705 Evaluation completed in {t_end - t_start:.1f}s")
print(f"  Total results: {n_results}")
print(f"  Failures: {n_failures}")

Run ID: notebook_live
Variants: title_only, vector_only, bm25_only, hybrid_union, hybrid_no_cold_boost
Fake results: False

Starting evaluation...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.05it/s]



✅ Evaluation completed in 69.5s
  Total results: 2428
  Failures: 0


In [8]:
# Layer 2: Variant Comparison Table
variant_summaries = run_data.get('variant_summaries', [])

print(f"\n{'='*100}")
print("VARIANT COMPARISON — Main Metrics")
print(f"{'='*100}")
print(f"{'Variant':<25} {'Queries':>8} {'NDCG@10':>10} {'Recall@10':>11} {'MRR@10':>9} {'Prec@5':>9} {'Hit@10':>9} {'ColdRel@10':>12}")
print("-" * 100)
for vs in variant_summaries:
    v = vs.get('variant', '')
    qc = vs.get('query_count', 0)
    ndcg = vs.get('ndcg_at_10')
    recall = vs.get('recall_at_10')
    mrr = vs.get('mrr_at_10')
    prec = vs.get('precision_at_5')
    hr = vs.get('hit_rate_at_10')
    cold = vs.get('cold_relevant_rate_at_10')
    
    def fmt(val):
        return f"{val:.4f}" if val is not None else "N/A"
    
    print(f"{v:<25} {qc:>8} {fmt(ndcg):>10} {fmt(recall):>11} {fmt(mrr):>9} {fmt(prec):>9} {fmt(hr):>9} {fmt(cold):>12}")
print(f"{'='*100}")


VARIANT COMPARISON — Main Metrics
Variant                    Queries    NDCG@10   Recall@10    MRR@10    Prec@5    Hit@10   ColdRel@10
----------------------------------------------------------------------------------------------------
title_only                      50     0.5530      0.3154    0.4992    0.3160    0.7400       0.3020
vector_only                     50     0.7191      0.4005    0.5537    0.4280    0.7400       0.3727
bm25_only                       50     0.6003      0.3257    0.5571    0.3440    0.7600       0.3342
hybrid_union                    50     0.7715      0.4525    0.6817    0.4800    0.7800       0.3940
hybrid_no_cold_boost            50     0.7715      0.4525    0.6817    0.4800    0.7800       0.3940


In [9]:
# Layer 2: Variant Availability
failures = run_data.get('failures', [])
failed_variants = {f.get('variant') for f in failures if f.get('error_type') == 'variant_unavailable'}

print("\nVariant Availability:")
print(f"{'Variant':<25} {'Status':<15} {'Included':>10}")
print("-" * 55)
for v in VARIANTS:
    if v in failed_variants:
        print(f"{v:<25} {'\u274c unavailable':<15} {'no':>10}")
    else:
        print(f"{v:<25} {'\u2705 available':<15} {'yes':>10}")


Variant Availability:
Variant                   Status            Included
-------------------------------------------------------
title_only                ✅ available            yes
vector_only               ✅ available            yes
bm25_only                 ✅ available            yes
hybrid_union              ✅ available            yes
hybrid_no_cold_boost      ✅ available            yes


In [10]:
# Layer 2: Slice-level breakdown
slice_summaries = run_data.get('slice_summaries', [])

if slice_summaries:
    # Group by variant, show top slices
    print(f"\n{'='*90}")
    print("SLICE BREAKDOWN — NDCG@10 by variant and slice")
    print(f"{'='*90}")
    print(f"{'Variant':<25} {'Slice':<25} {'Queries':>8} {'NDCG@10':>10} {'Recall@10':>11}")
    print("-" * 90)
    for ss in sorted(slice_summaries, key=lambda x: (str(x.get('variant', '')), str(x.get('slice', '')))):
        v = ss.get('variant', '')
        s = ss.get('slice', '')
        qc = ss.get('query_count', 0)
        ndcg = ss.get('ndcg_at_10')
        recall = ss.get('recall_at_10')
        ndcg_str = f"{ndcg:.4f}" if ndcg is not None else "N/A"
        recall_str = f"{recall:.4f}" if recall is not None else "N/A"
        print(f"{v:<25} {s:<25} {qc:>8} {ndcg_str:>10} {recall_str:>11}")
    print(f"{'='*90}")
else:
    print("No slice-level summaries available.")


SLICE BREAKDOWN — NDCG@10 by variant and slice
Variant                   Slice                      Queries    NDCG@10   Recall@10
------------------------------------------------------------------------------------------
bm25_only                 beauty                          26     0.6179      0.3821
bm25_only                 compatibility                    9     0.6964      0.2367
bm25_only                 constraint                       9     0.7140      0.4431
bm25_only                 electronics                     24     0.5812      0.2665
bm25_only                 english                         32     0.6426      0.3585
bm25_only                 gift                             4     0.4524      0.3003
bm25_only                 occasion                         7     0.3906      0.2484
bm25_only                 persona                         10     0.5855      0.3887
bm25_only                 price_filter                     8     0.2741      0.0833
bm25_only            

---

# Layer 3 — Demo Readiness & Claim Status

Mỗi claim là một tuyên bố chất lượng hệ thống. Status:
- **supported** = có evidence ủng hộ
- **unsupported** = evidence phản bác
- **needs_more_evidence** = chưa đủ data để kết luận


In [11]:
# Layer 3: Claim Status
claims = decide_claim_status(
    variant_summaries,
    run_data.get('config', {}),
    failures,
)

print(f"\n{'='*100}")
print("CLAIM STATUS — Demo Readiness")
print(f"{'='*100}")
print(f"{'Claim':<40} {'Status':<22} {'Evidence / Blocker'}")
print("-" * 100)
for c in claims:
    icon = '\u2705' if c.status == 'supported' else ('\u274c' if c.status == 'unsupported' else '\u26a0\ufe0f')
    info = c.evidence if c.evidence else c.blocker
    print(f"{c.claim:<40} {icon} {c.status:<19} {info}")
print(f"{'='*100}")


CLAIM STATUS — Demo Readiness
Claim                                    Status                 Evidence / Blocker
----------------------------------------------------------------------------------------------------
Hybrid beats title baseline              ⚠️ needs_more_evidence paired comparison evidence not supportive: recall_at_10 significance=directional_only (null_metric_pair_count 7)
Hybrid beats single-channel baselines    ✅ supported           hybrid=0.7715 > vector=0.7191, bm25=0.6003; paired NDCG@10 CI evidence positive
Cold-start exposure quality              ✅ supported           ColdRelevantRate@10=0.394, but dataset is cold-dominant — cannot prove cold vs warm lift
Cold-start window was measured           ⚠️ needs_more_evidence indexed_at and first_seen_in_top_k_at timestamps not available in this run
Vietnamese robustness                    ✅ supported           hybrid Vietnamese NDCG@10=0.8462 > title=0.4913; slice paired CI evidence positive
Price-filter retrieval quali

In [12]:
# Layer 3: Latency Summary
latency = run_data.get('latency', [])

if latency:
    search_lats = sorted([r['search_latency_ms'] for r in latency if 'search_latency_ms' in r])
    total_lats = sorted([r['total_latency_ms'] for r in latency if 'total_latency_ms' in r])
    
    print(f"\n{'='*60}")
    print("LATENCY SUMMARY")
    print(f"{'='*60}")
    print(f"Sample size: {len(search_lats)}")
    if search_lats:
        s_p50 = search_lats[len(search_lats) // 2]
        s_p95 = search_lats[min(int(len(search_lats) * 0.95), len(search_lats) - 1)]
        print(f"Search P50:  {s_p50:.1f}ms")
        print(f"Search P95:  {s_p95:.1f}ms")
    if total_lats:
        t_p50 = total_lats[len(total_lats) // 2]
        t_p95 = total_lats[min(int(len(total_lats) * 0.95), len(total_lats) - 1)]
        print(f"Total P50:   {t_p50:.1f}ms")
        print(f"Total P95:   {t_p95:.1f}ms")
    confidence = 'high' if len(search_lats) >= 50 else ('medium' if len(search_lats) >= 20 else 'low')
    print(f"Confidence:  {confidence}")
    print(f"{'='*60}")

    # Per-variant latency
    print(f"\nPer-variant average search latency:")
    print(f"{'Variant':<25} {'Avg (ms)':>10} {'Count':>8}")
    print("-" * 45)
    variant_lats = {}
    for r in latency:
        v = r.get('variant', '')
        variant_lats.setdefault(v, []).append(r['search_latency_ms'])
    for v, lats in sorted(variant_lats.items()):
        avg = sum(lats) / len(lats)
        print(f"{v:<25} {avg:>10.1f} {len(lats):>8}")
else:
    print("No latency data available.")


LATENCY SUMMARY
Sample size: 250
Search P50:  89.3ms
Search P95:  183.8ms
Total P50:   861.6ms
Total P95:   956.2ms
Confidence:  high

Per-variant average search latency:
Variant                     Avg (ms)    Count
---------------------------------------------
bm25_only                       89.6       50
hybrid_no_cold_boost           108.1       50
hybrid_union                   106.1       50
title_only                     189.4       50
vector_only                    110.3       50


In [13]:
# Layer 3: Failure Analysis
if failures:
    print(f"\n{'='*80}")
    print("FAILURE SUMMARY")
    print(f"{'='*80}")
    error_types = {}
    for f in failures:
        et = f.get('error_type', 'unknown')
        error_types[et] = error_types.get(et, 0) + 1
    
    print(f"{'Error Type':<35} {'Count':>8}")
    print("-" * 45)
    for et, cnt in sorted(error_types.items()):
        print(f"{et:<35} {cnt:>8}")
    
    print(f"\nTotal failures: {len(failures)}")
    recoverable = sum(1 for f in failures if f.get('recoverable', False))
    print(f"Recoverable: {recoverable}")
    print(f"Non-recoverable: {len(failures) - recoverable}")
    print(f"{'='*80}")
else:
    print("\u2705 No failures recorded.")

✅ No failures recorded.


---

# 💾 Save Report & Artifacts

Ghi tất cả outputs ra thư mục output. Bao gồm:
- `metrics_summary.md` — Markdown report đầy đủ
- `layer2_metrics_summary.json` — JSON cho CI/CD hoặc dashboard
- `layer2_metrics_by_query.csv` — Chi tiết per-query metrics
- `latency_by_query.csv` — Chi tiết latency
- `failures.json` — Danh sách lỗi
- `manifest.json` — Danh sách tất cả artifacts


In [14]:
# Save all outputs
paths = write_evaluation_outputs(run_data, OUTPUT_DIR)

print("\u2705 Artifacts written:")
print(f"{'Name':<25} {'Path'}")
print("-" * 70)
for name, path in sorted(paths.items()):
    print(f"{name:<25} {path}")

✅ Artifacts written:
Name                      Path
----------------------------------------------------------------------
config                    .runtime\evaluation\notebook_run\config.json
failures                  .runtime\evaluation\notebook_run\failures.json
latency                   .runtime\evaluation\notebook_run\latency_by_query.csv
manifest                  .runtime\evaluation\notebook_run\manifest.json
metrics_by_query          .runtime\evaluation\notebook_run\layer2_metrics_by_query.csv
metrics_summary_json      .runtime\evaluation\notebook_run\layer2_metrics_summary.json
metrics_summary_md        .runtime\evaluation\notebook_run\metrics_summary.md
raw_results               .runtime\evaluation\notebook_run\layer2_raw_results.json


In [15]:
# Display the full Markdown report
from IPython.display import Markdown, display

md_report = generate_metrics_summary_md(run_data)
display(Markdown(md_report))

# Evaluation Report

**Run ID:** notebook_live
**Created at:** 2026-05-29T20:04:51
**K values:** [1, 3, 5, 10]
**Relevance threshold:** 2
**Git commit:** 39c28bc
**Python version:** 3.14.0
**Platform:** Windows-11-10.0.26200-SP0

## Judgment Coverage

| Metric | Value | Gate | Status |
|--------|-------|------|--------|
| Total queries | 50 | — | — |
| Judged queries | 50 | ≥ 30 | ✅ |
| Positive judged queries | 43 | ≥ 20 | ✅ |
| Query coverage rate | 100.0% | ≥ 50% | ✅ |
| Result coverage rate | 99.6% | — | — |
| Report status | **sufficient** | — | — |

## Judgment Provenance

- **Overall audit status:** `ai_assisted`
- **Human-audited rate:** `0.0`

| Source | Count |
|---|---|
| ai_assisted | 2119 |

| Slice | Queries | Judged | Human-audited queries | Audit status | Demo readiness | Blockers |
|---|---|---|---|---|---|---|
| beauty | 26 | 26 | 0 | ai_assisted | directional_only | judgment_source_status is ai_assisted |
| compatibility | 9 | 9 | 0 | ai_assisted | directional_only | judgment_source_status is ai_assisted |
| constraint | 9 | 9 | 0 | ai_assisted | directional_only | judgment_source_status is ai_assisted |
| electronics | 24 | 24 | 0 | ai_assisted | directional_only | judgment_source_status is ai_assisted |
| english | 32 | 32 | 0 | ai_assisted | directional_only | judgment_source_status is ai_assisted |
| gift | 4 | 4 | 0 | ai_assisted | directional_only | judgment_source_status is ai_assisted |
| occasion | 7 | 7 | 0 | ai_assisted | directional_only | judgment_source_status is ai_assisted |
| persona | 10 | 10 | 0 | ai_assisted | directional_only | judgment_source_status is ai_assisted |
| price_filter | 8 | 8 | 0 | ai_assisted | directional_only | judgment_source_status is ai_assisted |
| problem | 5 | 5 | 0 | ai_assisted | directional_only | judgment_source_status is ai_assisted |
| vietnamese | 13 | 13 | 0 | ai_assisted | directional_only | judgment_source_status is ai_assisted |
| vietnamese_no_diacritic | 5 | 5 | 0 | ai_assisted | directional_only | judgment_source_status is ai_assisted |

## Cold/Warm Distribution

- **Cold items:** 300
- **Warm items:** 0
- **Unknown status:** 0
- **Cold percentage:** 100.0%

> ⚠️ **Dataset is cold-dominant** — Cold-start metrics measure exposure quality, not cold vs warm lift. Insufficient warm items for comparison.

## Variant Availability

| variant | availability | reason | recoverable | included_in_comparison |
|---|---|---|---|---|
| title_only | available | — | — | yes |
| vector_only | available | — | — | yes |
| bm25_only | available | — | — | yes |
| hybrid_union | available | — | — | yes |
| hybrid_no_cold_boost | available | — | — | yes |

## Variant Comparison

| variant | query_count | metric_confidence | NDCG@10 | Recall@10 | MRR@10 | Precision@5 | HitRate@10 | ColdRelevantRate@10 | empty_results | failures |
|---|---|---|---|---|---|---|---|---|---|---|
| title_only | 50 | high | 0.553 | 0.3154 | 0.4992 | 0.316 | 0.74 | 0.302 | 0 | 0 |
| vector_only | 50 | high | 0.7191 | 0.4005 | 0.5537 | 0.428 | 0.74 | 0.3727 | 0 | 0 |
| bm25_only | 50 | high | 0.6003 | 0.3257 | 0.5571 | 0.344 | 0.76 | 0.3342 | 1 | 0 |
| hybrid_union | 50 | high | 0.7715 | 0.4525 | 0.6817 | 0.48 | 0.78 | 0.394 | 0 | 0 |
| hybrid_no_cold_boost | 50 | high | 0.7715 | 0.4525 | 0.6817 | 0.48 | 0.78 | 0.394 | 0 | 0 |

## Ablation Impact

| comparison | delta_NDCG@10 | delta_MRR@10 | delta_ColdRelevantRate@10 | paired evidence | interpretation |
|---|---|---|---|---|---|
| hybrid vs title_only | +0.2185 | +0.1825 | +0.092 | positive | Clear improvement ✅ |
| hybrid vs vector_only | +0.0524 | +0.128 | +0.0213 | positive | Clear improvement ✅ |
| hybrid vs bm25_only | +0.1712 | +0.1246 | +0.0598 | positive | Clear improvement ✅ |
| hybrid vs no_cold_boost | 0.0 | 0.0 | 0.0 | directional_only | Directional only; do not treat as supported |

## Slice Analysis

### Language Slices

| Slice | Queries | Judged | Confidence | Hybrid NDCG@10 | Title NDCG@10 | Delta |
|-------|---------|--------|------------|----------------|---------------|-------|
| english | 32 | 32 | high  | 0.7538 | 0.6113 | +0.1425 |
| vietnamese | 13 | 13 | medium  | 0.8462 | 0.4913 | +0.3549 |
| vietnamese_no_diacritic | 5 | 5 | medium  | 0.691 | 0.3401 | +0.3509 |

### Intent Slices

| Slice | Queries | Judged | Confidence | Hybrid NDCG@10 | Best Variant |
|-------|---------|--------|------------|----------------|--------------|
| price_filter | 8 | 8 | medium  | 0.6528 | hybrid_union |
| compatibility | 9 | 9 | medium  | 0.8363 | hybrid_union |
| gift | 4 | 4 | low ⚠️ | 0.6778 | hybrid_union |
| persona | 10 | 10 | medium  | 0.7482 | hybrid_union |
| occasion | 7 | 7 | medium  | 0.6137 | hybrid_union |
| problem | 5 | 5 | medium  | 0.8711 | vector_only |
| constraint | 9 | 9 | medium  | 0.7538 | hybrid_union |

### Slice Confidence Diagnostics

| Slice | Variant | Confidence | Judged | Positive Judged | Result Coverage | Blockers |
|-------|---------|------------|--------|-----------------|-----------------|----------|
| english | title_only | high | 32 | 27 | 0.9968 | - |
| beauty | title_only | high | 26 | 19 | 0.9923 | - |
| constraint | title_only | medium | 9 | 7 | 0.9889 | - |
| english | vector_only | high | 32 | 27 | 1.0 | - |
| beauty | vector_only | high | 26 | 19 | 0.9921 | - |
| constraint | vector_only | medium | 9 | 7 | 1.0 | - |
| english | bm25_only | high | 32 | 26 | 1.0 | - |
| beauty | bm25_only | high | 26 | 19 | 1.0 | - |
| constraint | bm25_only | medium | 9 | 8 | 1.0 | - |
| english | hybrid_union | high | 32 | 28 | 1.0 | - |
| beauty | hybrid_union | high | 26 | 20 | 0.9922 | - |
| constraint | hybrid_union | medium | 9 | 8 | 1.0 | - |
| english | hybrid_no_cold_boost | high | 32 | 28 | 1.0 | - |
| beauty | hybrid_no_cold_boost | high | 26 | 20 | 0.9922 | - |
| constraint | hybrid_no_cold_boost | medium | 9 | 8 | 1.0 | - |
| problem | title_only | low | 5 | 2 | 0.98 | positive_judged_query_count 2 < 3 |
| problem | vector_only | medium | 5 | 3 | 1.0 | - |
| problem | bm25_only | low | 5 | 2 | 1.0 | positive_judged_query_count 2 < 3 |
| problem | hybrid_union | medium | 5 | 3 | 1.0 | - |
| problem | hybrid_no_cold_boost | medium | 5 | 3 | 1.0 | - |
| occasion | title_only | medium | 7 | 6 | 1.0 | - |
| occasion | vector_only | medium | 7 | 6 | 1.0 | - |
| occasion | bm25_only | medium | 7 | 5 | 1.0 | - |
| occasion | hybrid_union | medium | 7 | 6 | 1.0 | - |
| occasion | hybrid_no_cold_boost | medium | 7 | 6 | 1.0 | - |
| persona | title_only | medium | 10 | 9 | 0.99 | - |
| persona | vector_only | medium | 10 | 9 | 1.0 | - |
| persona | bm25_only | medium | 10 | 9 | 1.0 | - |
| persona | hybrid_union | medium | 10 | 9 | 1.0 | - |
| persona | hybrid_no_cold_boost | medium | 10 | 9 | 1.0 | - |
| electronics | title_only | high | 24 | 18 | 1.0 | - |
| compatibility | title_only | medium | 9 | 8 | 1.0 | - |
| electronics | vector_only | high | 24 | 18 | 1.0 | - |
| compatibility | vector_only | medium | 9 | 8 | 1.0 | - |
| electronics | bm25_only | high | 24 | 19 | 0.9955 | - |
| compatibility | bm25_only | medium | 9 | 9 | 0.9889 | - |
| electronics | hybrid_union | high | 24 | 19 | 0.9958 | - |
| compatibility | hybrid_union | medium | 9 | 8 | 0.9889 | - |
| electronics | hybrid_no_cold_boost | high | 24 | 19 | 0.9958 | - |
| compatibility | hybrid_no_cold_boost | medium | 9 | 8 | 0.9889 | - |
| vietnamese | title_only | medium | 13 | 8 | 0.9923 | - |
| vietnamese | vector_only | medium | 13 | 9 | 0.9836 | - |
| vietnamese | bm25_only | medium | 13 | 9 | 1.0 | - |
| vietnamese | hybrid_union | medium | 13 | 10 | 0.9844 | - |
| vietnamese | hybrid_no_cold_boost | medium | 13 | 10 | 0.9844 | - |
| vietnamese_no_diacritic | title_only | low | 5 | 2 | 1.0 | positive_judged_query_count 2 < 3 |
| vietnamese_no_diacritic | vector_only | low | 5 | 1 | 1.0 | positive_judged_query_count 1 < 3 |
| vietnamese_no_diacritic | bm25_only | medium | 5 | 3 | 0.9767 | - |
| vietnamese_no_diacritic | hybrid_union | low | 5 | 1 | 0.98 | positive_judged_query_count 1 < 3 |
| vietnamese_no_diacritic | hybrid_no_cold_boost | low | 5 | 1 | 0.98 | positive_judged_query_count 1 < 3 |
| price_filter | title_only | medium | 8 | 3 | 1.0 | - |
| price_filter | vector_only | low | 8 | 2 | 0.9688 | positive_judged_query_count 2 < 3 |
| price_filter | bm25_only | low | 8 | 1 | 1.0 | positive_judged_query_count 1 < 3 |
| price_filter | hybrid_union | medium | 8 | 3 | 0.973 | - |
| price_filter | hybrid_no_cold_boost | medium | 8 | 3 | 0.973 | - |
| gift | title_only | low | 4 | 4 | 1.0 | judged_query_count 4 < 5 |
| gift | vector_only | low | 4 | 4 | 1.0 | judged_query_count 4 < 5 |
| gift | bm25_only | low | 4 | 4 | 1.0 | judged_query_count 4 < 5 |
| gift | hybrid_union | low | 4 | 4 | 1.0 | judged_query_count 4 < 5 |
| gift | hybrid_no_cold_boost | low | 4 | 4 | 1.0 | judged_query_count 4 < 5 |

## Claim Status

| claim | status | evidence | blocker |
|---|---|---|---|
| Hybrid beats title baseline | ⚠️ needs_more_evidence |  | paired comparison evidence not supportive: recall_at_10 significance=directional_only (null_metric_pair_count 7) |
| Hybrid beats single-channel baselines | ✅ supported | hybrid=0.7715 > vector=0.7191, bm25=0.6003; paired NDCG@10 CI evidence positive |  |
| Cold-start exposure quality | ✅ supported | ColdRelevantRate@10=0.394, but dataset is cold-dominant — cannot prove cold vs warm lift |  |
| Cold-start window was measured | ⚠️ needs_more_evidence |  | indexed_at and first_seen_in_top_k_at timestamps not available in this run |
| Vietnamese robustness | ✅ supported | hybrid Vietnamese NDCG@10=0.8462 > title=0.4913; slice paired CI evidence positive |  |
| Price-filter retrieval quality | ✅ supported | hybrid price_filter NDCG@10=0.6528 > title=0.2836; slice paired CI evidence positive |  |
| Compatibility retrieval quality | ✅ supported | hybrid compatibility NDCG@10=0.8363 > title=0.6799; slice paired CI evidence positive |  |
| Live search latency evidence | ✅ supported | Live MongoDB with fresh fixtures; search P50=89.3ms, search P95=183.81ms, samples=250 |  |
| Live total path latency evidence | ✅ supported | Live MongoDB with fresh fixtures; total path P50=861.65ms, total path P95=956.16ms, samples=250 |  |

## What We Can Claim

- **Hybrid beats single-channel baselines:** hybrid=0.7715 > vector=0.7191, bm25=0.6003; paired NDCG@10 CI evidence positive
- **Cold-start exposure quality:** ColdRelevantRate@10=0.394, but dataset is cold-dominant — cannot prove cold vs warm lift
- **Vietnamese robustness:** hybrid Vietnamese NDCG@10=0.8462 > title=0.4913; slice paired CI evidence positive
- **Price-filter retrieval quality:** hybrid price_filter NDCG@10=0.6528 > title=0.2836; slice paired CI evidence positive
- **Compatibility retrieval quality:** hybrid compatibility NDCG@10=0.8363 > title=0.6799; slice paired CI evidence positive
- **Live search latency evidence:** Live MongoDB with fresh fixtures; search P50=89.3ms, search P95=183.81ms, samples=250
- **Live total path latency evidence:** Live MongoDB with fresh fixtures; total path P50=861.65ms, total path P95=956.16ms, samples=250

## What We Cannot Claim Yet

- **Hybrid beats title baseline:** needs_more_evidence; paired comparison evidence not supportive: recall_at_10 significance=directional_only (null_metric_pair_count 7)
- **Cold-start window was measured:** needs_more_evidence; indexed_at and first_seen_in_top_k_at timestamps not available in this run

## Claim Blockers

| claim | blocker |
|---|---|
| Hybrid beats title baseline | paired comparison evidence not supportive: recall_at_10 significance=directional_only (null_metric_pair_count 7) |
| Cold-start window was measured | indexed_at and first_seen_in_top_k_at timestamps not available in this run |

## Failure Dashboard

✅ No failures recorded.

## Latency

- **Sample size:** 250
- **P50 search latency:** 89.3ms
- **P95 search latency:** 183.8ms
- Search latency is not total path latency.
- **P50 total latency:** 861.6ms
- **P95 total latency:** 956.2ms
- Total path latency includes query processing plus search.
- **Latency confidence:** high

## Recommendations

- No automatic recommendations at this time.

## Explanation Coverage

- **Total results:** 2428
- **Has matched_intent:** 1378 (56.8%)
- **Has matched_fact:** 864 (35.6%)
- **Has both explanations:** 304 (12.5%)
- **Has neither explanation:** 490 (20.2%)
- **Explanation quality:** weak

> ⚠️ Less than 50% of results have full explanations (both matched_intent and matched_fact). This may affect demo readiness.

## Commands

### Commands Run

```bash
python scripts/run_evaluation.py \
  --queries evaluation/queries/retrieval_queries_seed.json \
  --judgments evaluation/judgments/retrieval_judgments_seed.json \
  --out .runtime/evaluation/notebook_run
```

### Commands Not Run

```bash
# Run diagnostics separately:
python scripts/run_eval_diagnostics.py --probes evaluation/queries/diagnostic_probes.json --out .runtime/evaluation/notebook_run/diagnostics

# Build judgment pool for labeling:
python scripts/build_eval_pool.py --queries evaluation/queries/retrieval_queries_seed.json --out .runtime/evaluation/notebook_run/pool --top-k 20
```


---

# 🔍 Deep Dive — Per-Query Analysis

Chọn 1 query để xem chi tiết kết quả từng variant và metrics.


In [16]:
# ============================================================
# Chọn query để deep dive
# ============================================================
DEEP_DIVE_QUERY_ID = "q001"  # ← ĐỔI Ở ĐÂY

# Find query info
target_query = next((q for q in queries if q.query_id == DEEP_DIVE_QUERY_ID), None)
if target_query is None:
    print(f"\u274c Query '{DEEP_DIVE_QUERY_ID}' not found. Available: {[q.query_id for q in queries[:10]]}...")
else:
    print(f"Query: {target_query.query_id}")
    print(f"Raw query: {target_query.raw_query}")
    print(f"Language: {target_query.language}")
    print(f"Topic: {target_query.topic}")
    print(f"Slices: {target_query.slices}")
    print(f"Intent tags: {target_query.intent_tags}")
    print(f"Expected filters: {target_query.expected_filters}")
    
    # Per-query metrics for this query
    per_q = run_data.get('per_query_metrics', [])
    query_rows = [r for r in per_q if r.get('query_id') == DEEP_DIVE_QUERY_ID]
    
    if query_rows:
        print(f"\n{'Variant':<25} {'NDCG@10':>10} {'MRR@10':>10} {'Results':>10} {'Judged':>8}")
        print("-" * 70)
        for r in query_rows:
            v = r.get('variant', '')
            ndcg = r.get('ndcg_at_10')
            mrr = r.get('mrr_at_10')
            rc = r.get('result_count', 0)
            jc = r.get('judged_result_count', 0)
            ndcg_str = f"{ndcg:.4f}" if ndcg is not None else "N/A"
            mrr_str = f"{mrr:.4f}" if mrr is not None else "N/A"
            print(f"{v:<25} {ndcg_str:>10} {mrr_str:>10} {rc:>10} {jc:>8}")
    
    # Show actual results for this query
    all_results = run_data.get('results', [])
    query_results = [r for r in all_results if r.get('query_id') == DEEP_DIVE_QUERY_ID]
    
    if query_results:
        print(f"\nTop results for {DEEP_DIVE_QUERY_ID}:")
        print(f"{'Variant':<25} {'Rank':>6} {'Item ID':<20} {'Score':>8} {'Title'}")
        print("-" * 100)
        for r in sorted(query_results, key=lambda x: (x.get('variant', ''), x.get('rank', 0)))[:20]:
            title = r.get('title', '')
            if len(title) > 35:
                title = title[:32] + '...'
            print(f"{r.get('variant', ''):<25} {r.get('rank', 0):>6} {r.get('item_id', ''):<20} {r.get('score', 0):>8.4f} {title}")

Query: q001
Raw query: moisturizing cream for dry skin
Language: en
Topic: skincare
Slices: ['english', 'beauty', 'constraint']
Intent tags: ['moisturizing', 'skin_type']
Expected filters: {'in_stock': True}

Variant                      NDCG@10     MRR@10    Results   Judged
----------------------------------------------------------------------
title_only                    0.7358     1.0000         10       10
vector_only                   0.5367     0.5000         10       10
bm25_only                     0.6193     0.5000         10       10
hybrid_union                  0.5791     0.5000         10       10
hybrid_no_cold_boost          0.5791     0.5000         10       10

Top results for q001:
Variant                     Rank Item ID                 Score Title
----------------------------------------------------------------------------------------------------
bm25_only                      1 B01N5P12CM             0.0557 Dove Men+Care Antiperspirant Deo...
bm25_only           

---

# 📋 Final Summary

Tổng kết toàn bộ evaluation run. Sử dụng bảng này làm checklist khi present demo.


In [17]:
# Final Summary Table
print(f"{'='*60}")
print("EVALUATION SUMMARY")
print(f"{'='*60}")
print(f"{'Metric':<35} {'Value'}")
print("-" * 60)
print(f"{'Run ID':<35} {config.run_id}")
print(f"{'Git commit':<35} {config.git_commit}")
print(f"{'Fake results':<35} {USE_FAKE_RESULTS}")
print(f"{'Queries evaluated':<35} {len(queries)}")
print(f"{'Judgments loaded':<35} {len(judgments)}")
print(f"{'Variants tested':<35} {len(VARIANTS)}")
print(f"{'Total results':<35} {n_results}")
print(f"{'Total failures':<35} {n_failures}")
print(f"{'Elapsed time':<35} {t_end - t_start:.1f}s")
print(f"{'Output directory':<35} {OUTPUT_DIR}")

# Layer 1 status
pr_str = f"{pr:.1%}" if pr is not None else 'N/A'
l1_icon = '\u2705' if diag_summary['failed'] == 0 else '\u274c'
print(f"{'Layer 1 pass rate':<35} {l1_icon} {pr_str} ({diag_summary['passed']}/{diag_summary['testable_probes']})")

# Layer 3 status
supported = sum(1 for c in claims if c.status == 'supported')
unsupported = sum(1 for c in claims if c.status == 'unsupported')
needs = sum(1 for c in claims if c.status == 'needs_more_evidence')
print(f"{'Claims supported':<35} {supported}")
print(f"{'Claims unsupported':<35} {unsupported}")
print(f"{'Claims need more evidence':<35} {needs}")

# Overall readiness
ready = (diag_summary['failed'] == 0 and n_failures == 0)
print(f"\n{'='*60}")
print(f"Pipeline status: {'\u2705 READY' if ready else '\u26a0\ufe0f NEEDS ATTENTION'}")
print(f"{'='*60}")

if not judgments:
    print("\n\u26a0\ufe0f Next step: Label relevance judgments to get real NDCG/Recall scores.")
    print("   1. Run: python scripts/build_eval_pool.py --queries evaluation/queries/retrieval_queries_seed.json --out .runtime/evaluation/pool")
    print("   2. Fill the relevance column in judgment_pool.csv (0-3).")
    print("   3. Import back: python scripts/import_eval_judgments.py --csv .runtime/evaluation/pool/judgment_pool.csv --out evaluation/judgments/retrieval_judgments_seed.json")

EVALUATION SUMMARY
Metric                              Value
------------------------------------------------------------
Run ID                              notebook_live
Git commit                          39c28bc
Fake results                        False
Queries evaluated                   50
Judgments loaded                    2119
Variants tested                     5
Total results                       2428
Total failures                      0
Elapsed time                        69.5s
Output directory                    .runtime/evaluation/notebook_run
Layer 1 pass rate                   ✅ 100.0% (14/14)
Claims supported                    7
Claims unsupported                  0
Claims need more evidence           2

Pipeline status: ✅ READY
